# Apply - Accident Workflow (end-user)

Given an `accident_id`, render four heatmaps:
1. Observed traffic in the accident window.
2. Historical baseline reconstructed from `historical_speed_profile_5m`.
3. Model prediction (any trained architecture).
4. Diff maps (observed - historical, observed - predicted).

Switching architectures is a single string change in the load cell. Hypothetical-accident input is a stub for round 2.

**Round-1 status:** observed + historical heatmaps work today; model loading depends on `scripts/lib/inference.py`, which becomes functional after `notebooks/train/*.ipynb` produces a checkpoint and `inference.load(...)` is wired.

In [ ]:
import os, sys
from pathlib import Path

# ── environment detection ────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _DEFAULT_ROOT = '/content/drive/MyDrive/陽交大高速公路計畫/main'
else:
    _here = Path(os.getcwd()).resolve()
    for _p in [_here, *_here.parents]:
        if (_p / 'CLAUDE.md').exists():
            _DEFAULT_ROOT = str(_p)
            break
    else:
        _DEFAULT_ROOT = str(_here)

PROJECT_ROOT = os.environ.get('PROJECT_ROOT', _DEFAULT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
%cd {PROJECT_ROOT}

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scripts.lib.calendar import day_class
from scripts.lib.data_loaders import load_samples, read_historical_profile, duckdb_connection
from scripts.lib.windows import SampleWindow, subset, absolute_mileage_axis_m
from scripts.lib.visualization import heatmap, diff_heatmap, compare_heatmaps

In [ ]:
# Pick an accident_id.
ACCIDENT_ID = '2024040004'  # change to inspect a different incident

samples = load_samples()
i = int(samples.metadata.index[samples.metadata['accident_id'] == ACCIDENT_ID][0])
row = samples.metadata.iloc[i].to_dict()
obs = samples.matrix[i]
window = samples.window
print(f"{row['accident_id']} | {row['freeway']}-{row['direction']} | {row['mileage_km']}K | report={row['report_time']} | filled_ratio={row['filled_ratio']:.2f}")

In [ ]:
heatmap(obs, window, accident=row, title='Observed')

## Historical baseline reconstruction

Look up `(vdid, day_class, peak_class, minute_bucket)` rows from `historical_speed_profile_5m` and place them on the same `(n_space, n_time)` grid.

In [ ]:
PEAKS = ((7*60, 9*60), (17*60, 19*60))

def peak_class(minute_of_day):
    for lo, hi in PEAKS:
        if lo <= minute_of_day < hi: return 'peak'
    return 'nonpeak'

report_time = pd.Timestamp(row['report_time'])
freeway = row['freeway']
direction = row['direction']
mileage_m = int(row['mileage_m'])
abs_mileage_m = absolute_mileage_axis_m(mileage_m, direction, window)

rows = []
for j, t_off in enumerate(window.time_offsets_min):
    ts_cell = report_time + pd.Timedelta(minutes=int(t_off))
    mod = ts_cell.hour * 60 + ts_cell.minute
    bucket = (mod // window.resolution_min) * window.resolution_min
    rows.append({
        'time_idx': j,
        'day_class': day_class(ts_cell),
        'peak_class': peak_class(mod),
        'minute_bucket': bucket,
    })
time_keys = pd.DataFrame(rows)
print(time_keys.head())

In [ ]:
# Resolve VDs whose mileage matches each space cell, then join to profile.
con = duckdb_connection()
vd_rows = con.sql(f"""
    SELECT mileage_m, mean_speed_x10/10.0 AS mean_speed,
           day_class, peak_class, minute_bucket
    FROM profile_5m
    WHERE freeway = '{freeway}' AND direction = '{direction}'
      AND mileage_m IN ({','.join(str(int(x)) for x in abs_mileage_m)})
""").df()

# Build the (n_space, n_time) baseline matrix.
baseline = np.full((window.n_space, window.n_time), np.nan, dtype=np.float32)
for s_idx, m in enumerate(abs_mileage_m):
    sub = vd_rows[vd_rows['mileage_m'] == int(m)]
    if sub.empty: continue
    merged = sub.merge(time_keys, on=['day_class', 'peak_class', 'minute_bucket'], how='inner')
    for _, mrow in merged.iterrows():
        baseline[s_idx, int(mrow['time_idx'])] = float(mrow['mean_speed'])

print('baseline NaN ratio =', float(np.isnan(baseline).mean()))

In [ ]:
compare_heatmaps([obs, baseline, obs - baseline], ['Observed', 'Historical baseline', 'Diff (obs - hist)'], window, accident=row)

## 模型預測（champion checkpoint）

使用 `reports/v1_champion.json` 的 champion（unet_feature_fusion / direct_concat），對當前事故給出預測。
NPZ 矩陣裁切至 champion 視窗（25+10 km，30+145 min → 36×36）後送入模型。

In [ ]:
from scripts.lib.inference import load, AccidentInput, ModelInput

model = load()          # loads v1 champion from reports/v1_champion.json
champ_win = model.window
print(f"Champion: {champ_win.upstream_km}+{champ_win.downstream_km} km, "
      f"{champ_win.time_before_min}+{champ_win.time_after_min} min → "
      f"{champ_win.n_space}×{champ_win.n_time}")

obs_sub, _ = subset(obs, window,
                    upstream_km=champ_win.upstream_km,
                    downstream_km=champ_win.downstream_km,
                    time_before_min=champ_win.time_before_min,
                    time_after_min=champ_win.time_after_min)
print(f"obs_sub: {obs_sub.shape}, "
      f"speed [{np.nanmin(obs_sub):.1f}, {np.nanmax(obs_sub):.1f}] km/h, "
      f"NaN ratio {np.isnan(obs_sub).mean():.1%}")

In [ ]:
_feat_path = Path(PROJECT_ROOT) / 'output' / 'feature_engineering' / 'incident_features_33.csv'
feat_df = pd.read_csv(_feat_path, dtype={'task_id': str})
feat_row = feat_df[feat_df['task_id'] == ACCIDENT_ID]
incident_features = feat_row.iloc[0].to_dict() if not feat_row.empty else {}
print(f"Features: lane_blockage_rate={incident_features.get('lane_blockage_rate', 'N/A')}, "
      f"accident_A2={incident_features.get('accident_A2', 'N/A')}, "
      f"injuries={incident_features.get('injuries', 'N/A')}")

accident_ai = AccidentInput(
    report_time=pd.Timestamp(row['report_time']).to_pydatetime(),
    freeway=row['freeway'],
    direction=row['direction'],
    mileage_km=row['mileage_km'],
)
mi = ModelInput(
    accident=accident_ai,
    window=champ_win,
    observed_pre=obs_sub,
    historical_baseline=np.full(obs_sub.shape, np.nan, dtype=np.float32),
    incident_features=incident_features,
    coverage_pre={},
    coverage_baseline={},
    fetch_status=None,
)

pred = model.predict_from_input(mi)
print(f"pred: {pred.shape}, speed [{np.nanmin(pred):.1f}, {np.nanmax(pred):.1f}] km/h")

In [ ]:
compare_heatmaps(
    [obs_sub, pred, obs_sub - pred],
    ['Observed (NPZ)', 'Predicted (champion t+15)', 'Error (obs − pred)'],
    champ_win,
    accident=row,
)

### 時間漸進預測（t+0 / t+5 / t+15）

同一事故在資訊逐步到位時的三個預測版本：
- **t+0**：僅初始通報（地點 + 靜態幾何 + 天候）
- **t+5**：加入事故類型（A1/A2/A3 + 特殊事故旗標）
- **t+15 champion**：加入傷亡人數與車道阻斷比例

In [ ]:
_T0_ZERO = frozenset({
    'injuries', 'fatalities', 'lane_blockage_rate',
    'special_accident', 'accident_A1', 'accident_A2', 'accident_A3',
})
_T5_ZERO = frozenset({'injuries', 'fatalities', 'lane_blockage_rate'})

def _mask_feats(feat, zero_cols):
    return {k: (0.0 if k in zero_cols else v) for k, v in feat.items()}

model_t0 = load("unet_feature_fusion", run_id="20260521T150042Z", experiment="direct_concat")
model_t5 = load("unet_feature_fusion", run_id="20260521T150048Z", experiment="direct_concat")

def _make_mi(feats):
    return ModelInput(
        accident=accident_ai, window=champ_win, observed_pre=obs_sub,
        historical_baseline=np.full(obs_sub.shape, np.nan, dtype=np.float32),
        incident_features=feats, coverage_pre={}, coverage_baseline={}, fetch_status=None,
    )

pred_t0 = model_t0.predict_from_input(_make_mi(_mask_feats(incident_features, _T0_ZERO)))
pred_t5 = model_t5.predict_from_input(_make_mi(_mask_feats(incident_features, _T5_ZERO)))

compare_heatmaps(
    [pred_t0, pred_t5, pred],
    ['t+0（初始通報）', 't+5（巡邏抵達）', 't+15 champion（完整資訊）'],
    champ_win,
    accident=row,
)

## Hypothetical accident (round 2 stub)

Let the user enter `freeway, direction, mileage_km, report_time, lanes_blocked` and reconstruct the same workflow without depending on `accident_data.csv`. Requires building a pseudo-`row` and computing baseline + features from inputs.